# Visual Analytics and Data Storytelling: Climate & CO2 Emissions

**Research Question:** "In this project, we aim to investigate whether global CO2 emissions influences regional and global temperatures over time using real-world data."

### 1. Problem Statement
Climate change is a pressing global issue, and greenhouse gas emissions are a primary driver. We want to visually explore the historical relationship between CO2 emissions and changing land temperatures across various cities and countries to better understand this impact.

### 2. Data Overview
We are using the Climate & Weather dataset alongside Global CO2 Emissions data to map historical trends, distributions, and correlations.

In [ ]:
# Download datasets required for this notebook
!pip install kaggle > /dev/null
import os

!kaggle datasets download -d michaelkratochvil/global-land-temperatures-by-city -p .
print('Datasets successfully downloaded or verified in the current directory.')

In [31]:
#imports and settings for visuals
import dash
from dash import dcc, html
import pandas as pd
import os
from scipy import stats
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import random
import base64

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)


In [32]:
input_file = "OWID_CB.csv"
output_file = "OWID_CB_Cleaned.csv"   
if not os.path.exists("OWID_CB_Cleaned.csv"):
    print("cleaned file not found cleaning data..")

    if os.path.exists("OWID_CB.csv"):
    # Download the dataset
        df = pd.read_csv(input_file)
        df.rename(columns={'REF_AREA_LABEL': 'Country', 'TIME_PERIOD': 'Year', 'OBS_VALUE': 'CO2'}, inplace=True)
        cols_to_remove = [
     "STRUCTURE", "STRUCTURE_ID", "ACTION", "FREQ", "INDICATOR", "INDICATOR_LABEL",
     "SEX", "SEX_LABEL", "AGE", "AGE_LABEL", "URBANISATION", "URBANISATION_LABEL", 
      "UNIT_MEASURE", "COMP_BREAKDOWN_1", "COMP_BREAKDOWN_1_LABEL", "COMP_BREAKDOWN_2",
      "COMP_BREAKDOWN_2_LABEL", "COMP_BREAKDOWN_3", "COMP_BREAKDOWN_3_LABEL",
      "DATABASE_ID", "DATABASE_ID_LABEL", "UNIT_TYPE_LABEL", "TIME_FORMAT", 
     "TIME_FORMAT_LABEL", "COMMENT_OBS", "OBS_STATUS", "OBS_STATUS_LABEL", 
     "OBS_CONF", "OBS_CONF_LABEL"
    ]
    
    df_cleaned = df.drop(columns=cols_to_remove, errors='ignore')
    
    df_cleaned.to_csv(output_file, index=False)
    print("data should be cleaned and saved to", output_file)
    if not os.path.exists("OWID_CB.csv"):
        print("input file not found please download the dataset and save it as OWID_CB.csv")
    
else:
    print("cleaned file found loading data..")

cleaned file found loading data..


In [33]:
file_name = "OWID_CB_Cleaned.csv"
try:
    df = pd.read_csv(file_name)
    print("File loaded successfully.")
    print(df.head())    
except FileNotFoundError:
    print("File not found.")

df['C'] = pd.to_numeric(df['CO2'], errors='coerce').fillna(0)
df['Year'] = pd.to_numeric(df['Year'], errors='coerce')

df_filtered = df[df['Year'] <= 2023]

global_trends = df_filtered.groupby('Year')['C'].agg(['mean', 'sum']).reset_index()
global_trends.rename(columns={'mean': 'Global_Average_CO2', 'sum': 'Total_CO2'}, inplace=True)

global_trends.to_csv("Global_CO2_Trends.csv", index=False)







File loaded successfully.
  FREQ_LABEL REF_AREA      Country UNIT_MEASURE_LABEL  Year  CO2  UNIT_MULT  \
0     Annual      AFG  Afghanistan      Tonnes of CO2  1750  0.0          6   
1     Annual      AFG  Afghanistan      Tonnes of CO2  1751  0.0          6   
2     Annual      AFG  Afghanistan      Tonnes of CO2  1752  0.0          6   
3     Annual      AFG  Afghanistan      Tonnes of CO2  1753  0.0          6   
4     Annual      AFG  Afghanistan      Tonnes of CO2  1754  0.0          6   

  UNIT_MULT_LABEL UNIT_TYPE  
0        Millions    NUMBER  
1        Millions    NUMBER  
2        Millions    NUMBER  
3        Millions    NUMBER  
4        Millions    NUMBER  


In [34]:
# run this once to load and clean the data, then we can use the cleaned data for visualizations and analysis
input_file = "GlobalLandTemperaturesByCity.csv"
output_file = "cleaned_global_land_temperatures_by_city.csv"
if not os.path.exists("cleaned_global_land_temperatures_by_city.csv"):
    def load_and_clean_data():
        script_dir = os.getcwd()
    # Download the dataset
        input_file = "GlobalLandTemperaturesByCity.csv"
        output_file = "cleaned_global_land_temperatures_by_city.csv"
        input_path = os.path.join(script_dir, input_file)
        output_path = os.path.join(script_dir, output_file)

        if not os.path.exists(input_path):
            print("error file not found")

            return
        print("cleaning data")

        df = pd.read_csv(input_path)

        df['dt'] = pd.to_datetime(df['dt'])

        df.dropna(subset=['AverageTemperature'], inplace=True)

        df['AverageTemperatureUncertainty'] = df['AverageTemperatureUncertainty'].fillna(df['AverageTemperatureUncertainty'].mean())

        def convert_coords(value):
            if pd.isna(value) or not isinstance(value, str):
                return value
            direction = value[-1]
            try:
                numeric_section = float(value[:-1])
                if direction in ['S', 'W']:
                    return -numeric_section
                return numeric_section
            except ValueError:
                return value
        df['Latitude'] = df['Latitude'].apply(convert_coords)
        df['Longitude'] = df['Longitude'].apply(convert_coords)

    #create derived features
    #will create 'year', 'month', and 'decade' to better tell the story
        df['year'] = df['dt'].dt.year
        df['month'] = df['dt'].dt.month
        df['decade'] = (df['year'] // 10) * 10
        print("derived features created")

    #outlier detection
        Q1 = df['AverageTemperature'].quantile(0.25)
        Q3 = df['AverageTemperature'].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        outliers = df[(df['AverageTemperature'] < lower_bound) | (df['AverageTemperature'] > upper_bound)]
        print(f"number of outliers detected: {len(outliers)}")



        df.sort_values(by=['City', 'dt'], inplace=True)
        print("data cleaned")
    

        df.to_csv(output_path, index=False)
        print("data cleaned and saved")
    if  not os.path.exists(output_file):
        print("cleaned file not found cleaning data..")      
    load_and_clean_data()
    if not os.path.exists(input_file):
            print("input file not found please download the dataset and save it as OWID_CB.csv")
    
    else:
        print("cleaned file found loading data..")


In [45]:
if not os.path.exists("Global_CO2_Trends.png"):

    df_plot = global_trends[global_trends['Global_Average_CO2'] > 1e6].copy()

    recent_history = df_plot[df_plot['Year'] >= 1970]
    slope, intercept, r_value, p_value, std_err = stats.linregress(recent_history['Year'], recent_history['Global_Average_CO2'])

    future_years = np.arange(df_plot['Year'].max() + 1, 2041)
    predicted_values = intercept + slope * future_years

    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(12, 6))

    sns.lineplot(data=df_plot, x='Year', y='Global_Average_CO2', linewidth=2.5, label='Historical Data', color='#2c3e50')

    plt.plot(future_years, predicted_values, color='red', linestyle='--', linewidth=2, label='Linear Trend Projection (to 2040)')


    plt.title("Global Average CO2 Emissions: History & Projection", fontsize=16, fontweight='bold')
    plt.xlabel("Year", fontsize=12)
    plt.ylabel("Global Average CO2 Emissions", fontsize=12)
    plt.legend()

    plt.axvline(x=df_plot['Year'].max(), color='gray', linestyle=':', alpha=0.5)
    plt.annotate('Projection Start', xy=(df_plot['Year'].max(), predicted_values[0]), 
             xytext=(df_plot['Year'].max()-30, predicted_values[0]*1.2),
             arrowprops=dict(facecolor='black', arrowstyle='->'))

    plt.tight_layout()
    plt.savefig("Global_CO2_Trends.png")
    plt.close()
else:    
    print("Visualization found..")

Visualization found..


Interpretation: Global average CO2 emissions over time have gone up steeply.

Importance: The average CO2 and average temperatures have gone up.

KeyInsight: The average amount of CO2 has gone up signifigantly.

Interpretation: The avergage temperature has been on a stead incline shwon by both the individual points and average line.

Importance: The temperature has been on a steady rise since the late 1800's.

KeyInsight: Overal temperatures have increased with a few key outliers.



In [36]:
#Bar chart
file_name = "OWID_CB_Cleaned.csv"
df = pd.read_csv(file_name)
df['C'] = pd.to_numeric(df['CO2'], errors='coerce').fillna(0)

def categorize_emissions(co2_value):
    if co2_value < 1000:
        return 'Low Emissions'
    elif co2_value < 10000:
        return 'Moderate Emissions'
    else:
        return 'High Emissions'

df['Emissions_Category'] = df['C'].apply(categorize_emissions)


plt.figure(figsize=(12, 6))
sns.set_theme(style="whitegrid")
sns.countplot(data=df[df['Year'] == 2023], x='Emissions_Category', order=['Low Emissions', 'Moderate Emissions', 'High Emissions'], palette='Set2')
plt.title("Emissions Categories in 2023", fontsize=16)
plt.xlabel("Emissions Category", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.tight_layout()
plt.savefig("Emissions_Categories_2023.png")
plt.close()

C:\Users\andre\AppData\Local\Temp\ipykernel_20876\3322858366.py:19: FutureWarning:



Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.




Interpretation: This visualization shows the count of countries grouped by their CO2 emissions levels.

Importance: This is important as is shows the highest contributors of CO2 emissions

Key insight: The global emission profile is skewed towards a few specfic countries

In [46]:
if not os.path.exists("CO2_Emission_Trends_Top5.png"):
    df_CO2_raw = pd.read_csv("OWID_CB_Cleaned.csv")
    df_CO2_raw['C'] = pd.to_numeric(df_CO2_raw['CO2'], errors='coerce').fillna(0)
    co2_yearly = df_CO2_raw.groupby('Year')['C'].sum().reset_index()
    co2_yearly.rename(columns={'C': 'Global_CO2_Emissions'}, inplace=True)

    df_temp_raw = pd.read_csv("cleaned_global_land_temperatures_by_city.csv")

# Get top 5 emitters by total historical CO2
    top_5_countries = df_CO2_raw.groupby('Country')['C'].sum().nlargest(5).index
    df_top5_co2 = df_CO2_raw[df_CO2_raw['Country'].isin(top_5_countries)]

    plt.figure(figsize=(12, 6))
    sns.lineplot(data=df_top5_co2, x='Year', y='C', hue='Country', linewidth=2)

    plt.title("CO2 Emission Trends: Top 5 Historical Emitters", fontsize=16)
    plt.xlabel("Year", fontsize=14)
    plt.ylabel("Annual CO2 (Tonnes)", fontsize=14)
    plt.legend(title='Country')
    plt.tight_layout()
    plt.savefig("CO2_Emission_Trends_Top5.png")
    plt.close()

else:
    print("Visualization found..")

Visualization found..


Interpretation: shows the realtionship between year, CO2 emissions and average temperature

Importance: It shows the correlation between emitions, years and temperatures

Insight: Took years after 1900 to show after industrialization, the deep red show a high correlation between temp, CO2 and year.

In [38]:
plt.figure(figsize=(12, 6))

sns.lineplot(data=global_trends, x='Year', y='Global_Average_CO2')# Adding Annotations

plt.annotate('Industrial Revolution', xy=(1850, 5), xytext=(1780, 50),

             arrowprops=dict(facecolor='black', arrowstyle='->'))

plt.annotate('Post-WWII Economic Boom', xy=(1945, 100), xytext=(1900, 200),

             arrowprops=dict(facecolor='black', arrowstyle='->'))



plt.title('Significant Turning Points in CO2 History')
plt.savefig("Significant_Turning_Points_CO2_History.png")
plt.close()

Interpretation: Shows both the industrial revolution and Post ww2 Economic Boom

Importance: It shows Important events that led to increased CO2 production

Insight: Shows the important events when it comes to CO2 production

In [47]:
file_name = "cleaned_global_land_temperatures_by_city.csv"
if not os.path.exists("Distribution_Global_Average_Temperatures.png"):
    df_temp = pd.read_csv(file_name)
    plt.figure(figsize=(12, 6))
    sns.histplot(df_temp['AverageTemperature'], bins=30, kde=True, color='teal')

    plt.title("Distribution of Global Average Temperatures (1743 - Present)", fontsize=16)
    plt.xlabel("Average Temperature (°C)", fontsize=14)
    plt.ylabel("Frequency", fontsize=14)
    plt.tight_layout()
    plt.savefig("Distribution_Global_Average_Temperatures.png")
    plt.close()
else:
    print("Visualization found..")

Visualization found..


In [40]:
file_name = "cleaned_global_land_temperatures_by_city.csv"
df = pd.read_csv(file_name)
#derived 5 year period
df['5_year_period'] = (df['year'] // 5) * 5
five_year_avg = df.groupby('5_year_period')['AverageTemperature'].mean().reset_index()
plt.figure(figsize=(12, 6))
sns.set_theme(style="whitegrid")

sns.scatterplot(data=five_year_avg, x='5_year_period', y='AverageTemperature', s = 60, alpha = 0.7)

sns.regplot(data=five_year_avg, x='5_year_period', y='AverageTemperature', scatter=False, color='gray', line_kws={'linewidth': 2})
plt.title("Average Temperature by 5 Year Period", fontsize=16)
plt.xlabel("5 Year Period", fontsize=12)
plt.ylabel("Average Temperature in degrees celcius", fontsize=12)
plt.tight_layout()
plt.savefig("Average_Temperature_5_Year_Period.png")
plt.close()

In [41]:


df = pd.read_csv("cleaned_global_land_temperatures_by_city.csv")
df.columns = df.columns.str.strip()

if 'Year' not in df.columns:
    if 'year' in df.columns:
        df.rename(columns={'year': 'Year'}, inplace=True)
    elif 'dt' in df.columns:
        df['dt'] = pd.to_datetime(df['dt'])
        df['Year'] = df['dt'].dt.year
    else:
        print("Could not find a year column. Available columns are:", df.columns.tolist())

temp_col_found = [c for c in df.columns if 'temp' in c.lower()]
if temp_col_found:
    df.rename(columns={temp_col_found[0]: 'AverageTemperature'}, inplace=True)
else:
    print("Could not find a temperature column!")

df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
df['AverageTemperature'] = pd.to_numeric(df['AverageTemperature'], errors='coerce')
df = df.dropna(subset=['AverageTemperature', 'Year'])

df['Century'] = (df['Year'] // 100) * 100


plt.figure(figsize=(12, 6))
sns.set_theme(style="whitegrid")


sns.boxplot(data=df, x='Century', y='AverageTemperature', 
            hue='Century', palette='coolwarm', legend=False)

plt.title('Temperature Spread and Outliers by Century', fontsize=16, fontweight='bold')
plt.xlabel('Century (Start Year)', fontsize=12)
plt.ylabel('Average Temperature (°C)', fontsize=12)

plt.tight_layout()
plt.savefig("Temperature_Spread_Outliers_Century.png")
plt.close()

Interpretation: shows the distribution of temperature across the different centuries to get a overall trend of the trajectory of the temperatures.

Importance: it proves that the shift is in the entire range of temperatures

Insight: there is a clear upward migration of the temperatures since the 1800s to the 2000's.

In [41]:

df_CO2_raw = pd.read_csv("OWID_CB_Cleaned.csv")
df_CO2_raw['C'] = pd.to_numeric(df_CO2_raw['CO2'], errors='coerce').fillna(0)
co2_yearly = df_CO2_raw.groupby('Year')['C'].sum().reset_index()
co2_yearly.rename(columns={'C': 'Global_CO2_Emissions'}, inplace=True)

df_temp_raw = pd.read_csv("cleaned_global_land_temperatures_by_city.csv")

df_temp_raw.columns = df_temp_raw.columns.str.replace(' ', '_').str.title()
if 'Averagetemperature' in df_temp_raw.columns:
    df_temp_raw.rename(columns={'Averagetemperature': 'Average_Temperature'}, inplace=True)

df_temp_raw = df_temp_raw.dropna(subset=['Average_Temperature'])

temp_yearly = df_temp_raw.groupby('Year')['Average_Temperature'].mean().reset_index()
temp_yearly.rename(columns={'Average_Temperature': 'Global_Average_Temperature'}, inplace=True)

combined = pd.merge(co2_yearly, temp_yearly, on='Year')

# filter for after when industrialization happened
modern_era = combined[combined['Year'] >= 1900]

plt.figure(figsize=(10, 8))

target_cols = ['Year', 'Global_CO2_Emissions', 'Global_Average_Temperature']
corr_matrix = modern_era[target_cols].corr()

sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5, vmin=-1, vmax=1, center=0)
plt.title("Correlation Matrix: Modern Era (1900 - Present)", fontsize=16)
plt.tight_layout()
plt.savefig("Correlation_Matrix_Modern_Era.png")
plt.close()


KeyboardInterrupt: 

In [ ]:



df_temp = pd.read_csv("cleaned_global_land_temperatures_by_city.csv")
df_co2 = pd.read_csv("OWID_CB_Cleaned.csv")


df_temp['dt'] = pd.to_datetime(df_temp['dt'])
df_temp['Year'] = df_temp['dt'].dt.year
df_temp['Temp_f'] = (df_temp['AverageTemperature'] * 9/5) + 32


df_co2['CO2'] = pd.to_numeric(df_co2['CO2'], errors='coerce').fillna(0)
global_co2_trend = df_co2.groupby('Year')['CO2'].sum().reset_index()
global_co2_trend.columns = ['Year', 'Global_CO2_Emissions']



top_10_list = df_co2.groupby('Country')['CO2'].sum().nlargest(10).index.tolist()
all_other_countries = [c for c in df_temp['Country'].unique() if c not in top_10_list]
random_10_list = random.sample(all_other_countries, 10)

def get_city_history(country_list):
    city_picks = df_temp[df_temp['Country'].isin(country_list)].groupby('Country')['City'].first().reset_index()
    return pd.merge(df_temp, city_picks, on=['Country', 'City'])


df_top10 = get_city_history(top_10_list)
df_top10 = pd.merge(df_top10, global_co2_trend, on='Year')
df_top10['Category'] = 'Top 10 Producers'

df_random = get_city_history(random_10_list)
df_random = pd.merge(df_random, global_co2_trend, on='Year')
df_random['Category'] = 'Random 10'


df_combined = pd.concat([df_top10, df_random]).sort_values(by='Year').reset_index(drop=True)

print("df_combined has been created! You can now run the Dashboard cell.")

def launch_full_dashboard(sections_list, interactive_fig, conclusion_text):
    app = dash.Dash(__name__)

    # 1. HEADER SECTION
    header_markdown = """
# Visual Analytics and Data Storytelling: Climate & CO2 Emissions

**Research Question:** "In this project, we aim to investigate whether global CO2 emissions influences regional and global temperatures over time using real-world data."

### 1. Problem Statement
Climate change is a pressing global issue, and greenhouse gas emissions are a primary driver. We want to visually explore the historical relationship between CO2 emissions and changing land temperatures across various cities and countries to better understand this impact.

### 2. Data Overview
We are using the Climate & Weather dataset alongside Global CO2 Emissions data to map historical trends, distributions, and correlations.
    """

    layout_children = [
        dcc.Markdown(children=header_markdown, style={'lineHeight': '1.6', 'textAlign': 'left'}),
        html.Hr(style={'margin': '30px 0', 'borderColor': '#ddd'})
    ]

    # 2. LOOP THROUGH PNG SECTIONS
    for section in sections_list:
        try:
            with open(section['image_path'], "rb") as f:
                img_base64 = base64.b64encode(f.read()).decode('utf-8')
            
            block = html.Div([
                html.H2(section['title'], style={'color': '#2c3e50', 'marginTop': '40px'}),
                html.Div(style={'textAlign': 'center'}, children=[
                    html.Img(
                        src=f"data:image/png;base64,{img_base64}",
                        style={'width': '100%', 'border': '1px solid #ddd', 'boxShadow': '2px 2px 12px rgba(0,0,0,0.1)', 'borderRadius': '5px'}
                    )
                ]),
                html.Div([dcc.Markdown(children=section['insights'])], 
                         style={'marginTop': '20px', 'padding': '15px', 'backgroundColor': '#f8f9fa', 'borderRadius': '5px'}),
                html.Hr(style={'margin': '50px 0', 'borderColor': '#eee'})
            ])
            layout_children.append(block)
        except FileNotFoundError:
            layout_children.append(html.H3(f"Missing File: {section['image_path']}", style={'color': 'red'}))

    # 3. INTERACTIVE SCATTER
    layout_children.append(html.H2("Interactive Exploration: Global Emissions vs Local Temp", style={'color': '#2c3e50'}))
    layout_children.append(dcc.Graph(figure=interactive_fig))
    layout_children.append(html.Hr(style={'margin': '50px 0', 'borderColor': '#eee'}))

    # 4. FINAL INSIGHTS & RECOMMENDATIONS
    layout_children.append(html.Div([
        dcc.Markdown(children=conclusion_text, style={'lineHeight': '1.8'})
    ], style={'padding': '30px', 'backgroundColor': '#e9ecef', 'borderRadius': '8px', 'marginBottom': '50px'}))

    app.layout = html.Div(style={
        'backgroundColor': 'white', 'color': '#2c3e50', 'padding': '40px', 
        'fontFamily': 'sans-serif', 'maxWidth': '1100px', 'margin': 'auto'
    }, children=layout_children)

    # Height adjusted to 1400 to account for the new conclusion text
    app.run(mode='inline', port=8053, height=1400)

df_combined has been created! You can now run the Dashboard cell.


In [ ]:
my_report_sections = [
    {
        "title": "Global Average CO2 Emissions Over Time",
        "image_path": "Global_CO2_Trends.png",
        "insights": """
**Interpretation:** Global average CO2 emissions over time have gone up steeply.
**Importance:** The average CO2 and average temperatures have gone up.
**Key Insight:** The average amount of CO2 has gone up significantly.
        """
    },
    {
        "title": "Emissions Categories in 2023",
        "image_path": "Emissions_Categories_2023.png",
        "insights": """
**Interpretation:** This visualization shows the count of countries grouped by their CO2 emissions levels.
**Importance:** This is important as it shows the highest contributors of CO2 emissions.
**Key Insight:** The global emission profile is skewed towards a few specific countries.
        """
    },
    {
        "title": "CO2 Emission Trends: Top 5 Historical Emitters",
        "image_path": "CO2_Emission_Trends_Top5.png",
        "insights": """
**Interpretation:** Global average CO2 emissions over time have gone up steeply.
**Importance:** The average CO2 and average temperatures have gone up.
**Key Insight:** The average amount of CO2 has gone up significantly.
        """
    },
    {
        "title": "Significant Turning Points in CO2 History",
        "image_path": "Significant_Turning_Points_CO2_History.png",
        "insights": """
**Interpretation:** Shows both the industrial revolution and Post WWII Economic Boom.
**Importance:** It shows important events that led to increased CO2 production.
**Insight:** Shows the important events when it comes to CO2 production.
        """
    },
    {
        "title": "Distribution of Global Average Temperatures",
        "image_path": "Distribution_Global_Average_Temperatures.png",
        "insights": """
**Interpretation:** This histogram shows the frequency of average temperatures recorded globally.
**Importance:** It establishes the historical "baseline" for global weather.
**Insight:** Most global temperature records cluster between 15°C and 25°C, with a thin tail of extreme cold.
        """
    },
    {
        "title": "Average Temperature by 5 Year Period",
        "image_path": "Average_Temperature_5_Year_Period.png",
        "insights": """
**Interpretation:** The average temperature has been on a steady incline shown by both the individual points and average line.
**Importance:** The temperature has been on a steady rise since the late 1800s.
**Key Insight:** Overall temperatures have increased with a few key outliers.
        """
    },
    {
        "title": "Temperature Spread and Outliers by Century",
        "image_path": "Temperature_Spread_Outliers_Century.png",
        "insights": """
**Interpretation:** Shows the distribution of temperature across the different centuries to get an overall trend of the trajectory of the temperatures.
**Importance:** It proves that the shift is in the entire range of temperatures.
**Insight:** There is a clear upward migration of the temperatures since the 1800s to the 2000s.
        """
    },
    {
        "title": "Correlation Matrix: Modern Era (1900 - Present)",
        "image_path": "Correlation_Matrix_Modern_Era.png",
        "insights": """
**Interpretation:** Shows the relationship between year, CO2 emissions and average temperature.
**Importance:** It shows the correlation between emissions, years and temperatures.
**Insight:** Took years after 1900 to show; after industrialization, the deep red shows a high correlation between temp, CO2 and year.
        """
    }
]
# The text for your new sections
conclusion_markdown = """
## 4. Key Insights
* **Correlation:** The interactive scatter plot illustrates a strong, positive correlation between the increase in global CO2 emissions and climbing average temperatures over time.
* **Geographical Impact:** Over the decades, top-producing countries show faster and steeper temperature changes compared to many random regions, though the global upward trend affects everyone.
* **Historical Mapping:** Using a temporal mapping, we can visually capture how global temperature anomalies map tightly to periods of highest historical CO2 volume emissions.

## 5. Decision Recommendations
* **Targeted Emission Caps:** Since the top producers contribute disproportionately to the global CO2 levels, international policies should focus heavily on enforcing emission caps in these specific countries.
* **Accelerate Green Energy Transitions:** We recommend transitioning to sustainable, non-carbon-based energy in countries with the steepest historical acceleration in CO2 production.
* **Adaptive Infrastructure in Vulnerable Regions:** Both high and low emitters show warming patterns. Countries not producing massive emissions must still prepare adaptive infrastructure to cope with changing temporal weather patterns and rising foundational temperatures.
"""



fig_interactive = px.scatter(
    df_combined, 
    x="Global_CO2_Emissions", 
    y="Temp_f",
    animation_frame="Year", 
    animation_group="City", 
    color="Country",
    title="Interactive Exploration: Global CO2 vs Local Temperature",
    range_x=[df_combined['Global_CO2_Emissions'].min(), df_combined['Global_CO2_Emissions'].max() * 1.1],
    range_y=[df_combined['Temp_f'].min() - 5, df_combined['Temp_f'].max() + 5]
)
fig_interactive.update_layout(height=600, template="plotly_white", margin=dict(l=20, r=20, t=50, b=10))


launch_full_dashboard(my_report_sections, fig_interactive, conclusion_markdown)

### 4. Key Insights
- The interactive scatter plot illustrates a strong, positive correlation between the increase in global CO2 emissions and climbing average temperatures over time. 
- Over the decades, top-producing countries show faster and steeper temperature changes compared to many random regions, though the global upward trend affects everyone. 
- Using a temporal mapping, we can visually capture how global temperature anomalies map tightly to periods of highest historical CO2 volume emissions.
<br>

### 5. Decision Recommendations
Given the insights derived from this analysis, policy recommendations include:
1. **Targeted Emission Caps:** Since the top producers contribute disproportionately to the global CO2 levels, international policies should focus heavily on enforcing emission caps in these specific countries.
2. **Accelerate Green Energy Transitions:** We recommend transitioning to sustainable, non-carbon-based energy in countries with the steepest historical acceleration in CO2 production.
3. **Adaptive Infrastructure in Vulnerable Regions:** Both high and low emitters show warming patterns. Countries not producing massive emissions must still prepare adaptive infrastructure to cope with changing temporal weather patterns and rising foundational temperatures.